# Project — modelling notebook

**Author:** Andrea Gutiérrez Arcas** · **Dataset:**  [NBA Dataset: Box Scores and Stats (1947 - Today)](https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores) · **Last run:** *PENDIENTE*



## 0. Executive summary

> 📝 **Write this last.** Five lines for the stakeholder, no jargon:
> 1. The question the model answers, and for whom.
> 2. The model you chose, in one phrase.
> 3. How much better it is than today's rule, in the stakeholder's units.
> 4. What the stakeholder should do with it.
> 5. The main limit: when not to trust it.

*TODO*

## Setup

In [1]:
##Al ejecutar los import me ha dado error porque no tengo instalada la librería plotly.

%pip install plotly scikit-learn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import joblib  # noqa: F401
import numpy as np  # noqa: F401
import pandas as pd
import plotly.express as px  # noqa: F401
import sklearn
from sklearn.compose import ColumnTransformer  # noqa: F401
from sklearn.dummy import DummyClassifier, DummyRegressor  # noqa: F401
from sklearn.ensemble import (  # noqa: F401
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer  # noqa: F401
from sklearn.inspection import permutation_importance  # noqa: F401
from sklearn.linear_model import LinearRegression, LogisticRegression  # noqa: F401
from sklearn.metrics import (  # noqa: F401
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (  # noqa: F401
    GridSearchCV,
    RandomizedSearchCV,
    TimeSeriesSplit,
    cross_val_score,
)
from sklearn.pipeline import Pipeline  # noqa: F401
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # noqa: F401

# Use RANDOM_STATE wherever a random_state exists, so two runs give the same numbers.
RANDOM_STATE = 42
# Relative to this notebook (project/), or a URL string.
DATA_PATH = Path("data") / "nba_load.parquet" #ruta a mi notebook de limpieza
MODEL_PATH = Path("model.joblib")

print("pandas", pd.__version__, "· scikit-learn", sklearn.__version__)

pandas 3.0.1 · scikit-learn 1.9.1


## 1. From the brief to an ML task


| Question | Your answer |
|---|---|
| Decision the prediction feeds | *Cuántos minutos juega cada jugador de los Boston Celtics en el próximo partido: con normalidad, con límite de minutos o descanso* |
| Task type (regression / classification) | *Clasificación binaria: lesión nueva sí/no* |
| Target: exact column or formula | *`lesion_nueva` = 1 si el jugador jugó el partido anterior y se pierde el siguiente por lesión (`comment` clasificado como lesión); 0 en caso contrario* |
| Grain: what is one prediction? | *Un jugador antes de un partido* |
| When is the prediction made, and how far ahead? | *La víspera de cada partido, solo para ese partido. Tras jugarlo, sus datos se añaden y se predice el siguiente* |
| Features available at that moment | *Carga de los 14 días previos (hasta el último partido jugado), minutos del último partido, días de descanso hasta el próximo partido, si es back-to-back, edad, si es titular habitual y lesiones anteriores en la temporada* |
| Today's rule (the baseline to beat) | *El semáforo del brief: alerta si la carga de 14 días supera el percentil 80 del propio jugador* |
| Model metric, and why | *Recall y PR-AUC. Las lesiones nuevas son raras, así que la *accuracy* engaña. Dejar jugar a alguien que se lesiona cuesta más que un descanso de más, pero los descansos también tienen coste (victorias, normativa)* |
| Business translation (which KPI moves) | *KPI de ausencias por lesión y de disponibilidad de titulares* |

**Changes since the brief:**
- **El horizonte de la predicción.** El brief planteaba la carga de 14 días como semáforo. Ahora el modelo predice el riesgo para el **próximo partido** y se actualiza después de cada uno, que es como el responsable de rendimiento toma la decisión.
- **La variable objetivo es una lesión *nueva*.** Si el jugador ya estaba lesionado, predecir que no jugará no aporta nada al responsable de rendimiento. Esas filas se excluyen del entrenamiento.
- **Entreno con toda la liga y evalúo en los Celtics.** Las lesiones nuevas por partido son poco frecuentes, y con un solo equipo habría pocos casos para aprender. El modelo se entrena con los partidos de todos los equipos y se evalúa por separado en los Celtics, que siguen siendo el club piloto. Los datos se separan por tiempo: se entrena con las temporadas antiguas y se prueba con las más recientes.
- **De predecir minutos a predecir riesgo.** El objetivo inicial era recomendar cuántos minutos debe jugar cada jugador. Los datos solo muestran los minutos que dio el entrenador, no los que habrían sido mejores, así que un modelo de minutos aprendería a copiarle. Por eso el modelo predice el **riesgo de lesión**, y una regla de decisión lo convierte en una recomendación de minutos (normal / limitar / descansar).

**What this means for the stakeholder.** 
Antes de cada partido, la herramienta estima el riesgo de lesión de cada jugador y lo traduce en una recomendación de minutos: jugar con normalidad, limitar minutos o descansar. El objetivo es llegar con la plantilla sana a los playoffs, que es donde se decide el título. El modelo aprende de lo que les ha pasado a los jugadores de toda la liga, pero se aplica y se evalúa en la plantilla de los Celtics, y se actualiza después de cada partido.

## 2. Data and quality

**Ficha del dataset:** [NBA Dataset: Box Scores and Stats (1947 - Today)](https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores), publicado en Kaggle por Eoin A. Moore con datos de NBA.com, licencia CC0 (dominio público). Versión 515, descargada el 14/10/2026. **Grano:** una fila por jugador y partido. **Periodo:** desde 1947 hasta la temporada 2025-26; yo uso desde la temporada 2015-16.

La limpieza está en `notebooks/02-limpieza.ipynb`, que parte de los CSV originales (no están en GitHub porque pesan 390 MB) y guarda `data/nba_load.parquet` (3,9 MB). Aquí cargo ese archivo y compruebo que está como espero.

In [21]:
#PRIMERO: cargo el dataset limpio que guardé en 02-limpieza
df = pd.read_parquet(DATA_PATH)
print(f"Filas: {len(df):,} · Columnas: {df.shape[1]}")
print("Periodo:", df["gameDate"].min().date(), "→", df["gameDate"].max().date())

#SEGUNDO: compruebo que no hay jugadores repetidos en el mismo partido
#Cada jugador solo puede aparecer una vez por partido; si aparece dos veces, su carga se contaría doble
duplicated_player_game = df.duplicated(subset=["personId", "gameId"]).sum()
print("Filas duplicadas (mismo jugador y partido):", duplicated_player_game)

#TERCERO: nulos que quedan en cada columna
null_counts = df.isna().sum()
print("\nNulos por columna (solo las que tienen alguno):")
print(null_counts[null_counts > 0])

#CUARTO: tipos de datos, para comprobar que el parquet ha conservado fechas y números
df.dtypes

Filas: 366,218 · Columnas: 19
Periodo: 2015-10-27 → 2026-06-13
Filas duplicadas (mismo jugador y partido): 0

Nulos por columna (solo las que tienen alguno):
startingPosition    214516
comment             307623
edad                    11
dtype: int64


personId                        int64
firstName                         str
lastName                          str
gameId                          int64
gameDate               datetime64[us]
gameType                          str
season_start                    int32
comments_available               bool
playerteamName                    str
opponentteamName                  str
home                          float64
numMinutes                    float64
startingPosition                  str
points                        float64
fieldGoalsAttempted           float64
freeThrowsAttempted           float64
comment                           str
motivo_ausencia                   str
edad                          float64
dtype: object

**De 1.669.922 filas a 366.218.** Cada paso, con sus filas y el motivo:

| Step | Rows affected | Decision and reason |
|---|---|---|
| Partidos anteriores a la temporada 2015-16 | 1.273.392 eliminadas | Fuera de mi ventana de trabajo |
| Pretemporada, All-Star y otros | 30.312 eliminadas | No son competición real. Mantengo temporada regular, playoffs, Play-in y NBA Cup porque son oficiales y generan carga |
| Filas sin jugador (`personId` vacío) | 0 | Control: no se pueden asignar a ningún jugador |
| Minutos en formato mm:ss | 0 | Control: se convertirían a decimal |
| Duplicados (mismo jugador y partido) | 0 | Comprobado al cargar: cada jugador aparece una vez por partido |
| Minutos vacíos (no jugó) | 64.565 rellenadas con 0 | Ese día su carga fue 0; la media inventaría minutos y borrarlas quitaría las ausencias |
| Valores atípicos en minutos | 0 | El máximo (64 min) es posible: un partido con tres prórrogas. No hay negativos |
| Comentarios clasificados en `motivo_ausencia` | 58.595 | Por palabras clave: lesion, decision_tecnica u otros. lesion es la base del target |
| Protocolo COVID fuera de lesion | 67 | Un contagio no depende de la carga |
| Sin minutos y sin comentario (`sin_motivo`) | 8.392 | No jugó y no se sabe por qué: ni lesión ni partido jugado |
| Temporada 2021-22 sin comentarios | 33.848 marcadas | Se mantienen para la carga y H2, pero no se usan como target |
| Sin fecha de nacimiento | 11 | Se mantienen; el pipeline rellenará la edad con la mediana |

**What this means for the stakeholder.** Trabajo con diez temporadas completas de la NBA (2015-16 a 2025-26): más de 366.000 registros de jugador y partido, con los minutos jugados y, cuando un jugador no juega, el motivo. Los datos son fiables para medir la carga de minutos de cada jugador. Para saber quién se lesionó hay una limitación: en la temporada 2021-22 la fuente no guarda el motivo de las ausencias, así que esa temporada no la uso para aprender qué provoca las lesiones.

## 3. What the exploration showed

### Construcción del target

Antes de explorar necesito el target. Lo construyo aquí, y no con las features de la §4, porque forma parte de la definición del problema y la exploración lo necesita.

Para construirlo he tenido que tomar cuatro decisiones:

**Decisión 1: solo cuento las lesiones nuevas**

Cuando un jugador se lesiona, aparece como "lesion" en todos los partidos que se pierde. Si se pierde 10 partidos por un esguince, tiene 10 filas "lesion", pero solo la primera es el momento en que se lesiona. Las otras 9 son la misma lesión que continúa.

- **Alternativa descartada: usar todas las filas "lesion" como target.** El modelo aprendería que "si ayer estaba lesionado, hoy también", que acertaría mucho pero no serviría para nada: el cuerpo técnico ya sabe que un jugador lesionado no puede jugar. Lo que necesita saber es quién está a punto de lesionarse.
- **Decisión:** `lesion_nueva` = 1 si el jugador jugó el partido anterior y se pierde este por lesión.

**Decisión 2: solo entran en el modelo los jugadores que jugaron el partido anterior**

La pregunta "¿se lesionará en el próximo partido?" solo tiene sentido para un jugador que está sano y jugando. Si en el partido anterior ya estaba lesionado, descansó o no jugó por otro motivo, no está en riesgo de lesionarse jugando.

- **Alternativa descartada: incluir todas las filas y poner 0 en las que no son lesión nueva.** Mezclaría jugadores sanos con jugadores que ya estaban lesionados o no jugaron, y el modelo aprendería de casos en los que la pregunta no tiene sentido.
- **Decisión:** las filas en las que el jugador no jugó el partido anterior se quedan en el dataset (sus minutos cuentan para calcular la carga), pero el target queda vacío y no entran en el modelo.
- **Limitación:** una lesión que llega justo después de un descanso no cuenta como nueva. Lo acepto porque mi modelo mide la carga de los minutos jugados, y en ese caso el último partido no lo jugó.

**Decisión 3: el partido anterior tiene que ser de la misma temporada**

En el primer partido de cada temporada, el "partido anterior" fue meses antes, en la temporada pasada.

- **Alternativa descartada: contar también el primer partido de la temporada.** Una lesión en ese partido puede venir del verano o de la pretemporada, no de la carga de minutos, y la carga de los últimos 14 días sería 0 porque no ha habido partidos. Serían casos que mi modelo no puede explicar.
- **Decisión:** solo cuento como en riesgo las filas cuyo partido anterior es de la misma temporada.

**Decisión 4: la temporada 2021-22 no tiene target**

En la limpieza vi que esa temporada no tiene comentarios, así que no se sabe quién se lesionó (ver `02-limpieza.ipynb`).

- **Alternativa descartada: poner 0 en todas sus filas.** Sería decir que en toda una temporada nadie se lesionó, y el modelo aprendería algo falso.
- **Decisión:** sus filas se quedan para calcular la carga, pero no tienen target (`comments_available = False`).

**En resumen:** una fila está **en riesgo** (y entra en el modelo) si el jugador jugó el partido anterior, ese partido es de la misma temporada y la temporada tiene comentarios. Las demás filas se quedan en el dataset para calcular la carga, pero con el target vacío.

In [23]:
#PRIMERO: ordeno por jugador y fecha para que "el partido anterior" sea el anterior de ese jugador
df = df.sort_values(["personId", "gameDate"]).reset_index(drop=True)

#SEGUNDO: para cada fila, miro qué pasó en el partido anterior del mismo jugador
#shift(1) dentro de cada jugador trae el valor de su fila anterior; en su primer partido queda vacío
by_player = df.groupby("personId")
df["prev_status"] = by_player["motivo_ausencia"].shift(1)
df["prev_season"] = by_player["season_start"].shift(1)

#TERCERO: condiciones para estar "en riesgo"
played_prev_game = df["prev_status"] == "jugó"
same_season = df["prev_season"] == df["season_start"]
df["at_risk"] = played_prev_game & same_season & df["comments_available"]

#CUARTO: el target, solo en las filas en riesgo (en las demás queda vacío)
df["lesion_nueva"] = (df["motivo_ausencia"] == "lesion").astype("Int64").where(df["at_risk"])

#QUINTO: compruebo por qué queda fuera cada fila (los cinco grupos deben sumar el total)
exclusion_summary = pd.Series({
    "Primer partido del jugador en el dataset": df["prev_status"].isna().sum(),
    "No jugó el partido anterior": (df["prev_status"].notna() & ~played_prev_game).sum(),
    "Partido anterior en otra temporada": (played_prev_game & ~same_season).sum(),
    "Temporada sin comentarios (2021-22)": (played_prev_game & same_season & ~df["comments_available"]).sum(),
    "En riesgo: filas del modelo": df["at_risk"].sum(),
})
print(exclusion_summary)
print("Suma:", exclusion_summary.sum(), "| Total filas:", len(df))

#SEXTO: compruebo la decisión 1: qué pasó en el partido anterior de cada fila "lesion"
print("\nPartido anterior de las filas 'lesion':")
print(df.loc[df["motivo_ausencia"] == "lesion", "prev_status"].value_counts(dropna=False))

print("\nTarget:")
print(df["lesion_nueva"].value_counts(dropna=False))

#Porcentaje de lesiones nuevas: lesiones nuevas / filas en riesgo
new_injuries = int(df["lesion_nueva"].sum())       #cuántos 1 hay (sum ignora los vacíos)
rows_at_risk = int(df["at_risk"].sum())            #cuántas filas tienen target
injury_rate = new_injuries / rows_at_risk
print(f"Lesiones nuevas: {new_injuries:,} de {rows_at_risk:,} filas en riesgo = {injury_rate:.2%}")


Primer partido del jugador en el dataset      1582
No jugó el partido anterior                  66335
Partido anterior en otra temporada            3339
Temporada sin comentarios (2021-22)          27363
En riesgo: filas del modelo                 267599
dtype: int64
Suma: 366218 | Total filas: 366218

Partido anterior de las filas 'lesion':
prev_status
lesion              4050
jugó                3564
decision_tecnica     380
otros                 62
sin_motivo            33
NaN                   24
Name: count, dtype: int64

Target:
lesion_nueva
0       264081
<NA>     98619
1         3518
Name: count, dtype: Int64
Lesiones nuevas: 3,518 de 267,599 filas en riesgo = 1.31%


**Resultado:** de 366.218 filas, 267.599 están en riesgo y forman la tabla del modelo. Hay 3.518 lesiones nuevas, el 1,31 % de las filas en riesgo.

- La decisión de contar solo las lesiones nuevas era necesaria: de las 8.113 filas "lesion", 4.050 vienen después de otra fila "lesion". Son la misma lesión que continúa. Si las contara, el modelo aprendería que "si ayer estaba lesionado, hoy también", que no sirve para decidir nada.
- **Limitación:** hay 380 lesiones que llegan después de un descanso del entrenador. No cuentan como nuevas porque el jugador no jugó el partido anterior, y mi modelo mide la carga de minutos jugados.
- **Con un 1,31 % de positivos, la clase está muy desbalanceada.** Un modelo que dijera siempre "no se lesiona" acertaría el 98,7 % de las veces y no serviría para nada. Por eso uso recall y PR-AUC, y no accuracy, como decidí en la §1.

In [24]:
#Ejemplo: la racha de partidos seguidos con lesión más larga de un jugador
#Cada vez que el motivo cambia, empieza una "racha" nueva; cuento cuántos partidos dura cada una
injury_row = df["motivo_ausencia"] == "lesion"
streak_id = (df["motivo_ausencia"] != df["prev_status"]).cumsum()
streak_length = injury_row.groupby(streak_id).transform("sum")

#Me quedo con la racha de lesión más larga y enseño sus partidos, más el anterior (cuando aún jugaba)
longest = streak_id[injury_row & (streak_length == streak_length[injury_row].max())].iloc[0]
rows = df.index[streak_id == longest]
example = df.loc[rows.min() - 1 : rows.max(), ["firstName", "lastName", "gameDate", "motivo_ausencia", "prev_status", "lesion_nueva"]]
print(f"Racha de lesión más larga: {len(rows)} partidos")
example.head(6)

Racha de lesión más larga: 42 partidos


,firstName,lastName,gameDate,motivo_ausencia,prev_status,lesion_nueva
213918,Zach,Collins,2020-08-15 14:30:00,jugó,jugó,0
213919,Zach,Collins,2020-08-18 21:00:00,lesion,jugó,1
213920,Zach,Collins,2020-08-20 21:00:00,lesion,lesion,<NA>
213921,Zach,Collins,2020-08-22 20:30:00,lesion,lesion,<NA>
213922,Zach,Collins,2020-08-24 21:00:00,lesion,lesion,<NA>
213923,Zach,Collins,2020-08-29 21:00:00,lesion,lesion,<NA>


**Ejemplo:** en la racha de lesión más larga del dataset, el jugador se pierde 42 partidos seguidos. Solo el primero cuenta como lesión nueva (el partido anterior lo jugó); en los demás, el partido anterior también fue "lesion", así que es la misma lesión que continúa y no entra en el modelo.

| Hypothesis (from the brief) | Status: supported / refuted / not testable yet | Evidence (chart, table or number) |
|---|---|---|
| H1 — *TODO* | | |
| H2 — *TODO* | | |

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 4. Features

> 📝 Build the features (lesson 05: lags, rolling means, date parts, group aggregates,
> encodings) and fill in the feature table. Then the **leakage check**: for every feature, would
> its value be known at the moment the prediction is made?

In [7]:
# TODO engineer the features
# df["month"] = df["TODO_date_column"].dt.month
# df["TODO_lag1"] = df.groupby("TODO_key")["TODO_target"].shift(1)

In [8]:
# TODO the final lists the pipeline will use
TARGET = "TODO_target"
NUMERIC = []  # e.g. ["TODO_lag1", "month"]
CATEGORICAL = []  # e.g. ["TODO_district"]

| Feature | How it is built | Why it should help | Known at prediction time? |
|---|---|---|---|
| *TODO* | | | yes / no → dropped |

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 5. Split and baseline

> 📝 If your data has dates, split in time: train on the past, validate on the next period, and
> keep the most recent period as the **test set, untouched until section 8**. Score today's
> rule and a naive baseline (mean, median or majority class) on the validation data: every
> model from now on is judged against them.

In [9]:
# TODO the split — dates you choose and why
# train = df[df["TODO_date_column"] < "TODO"]
# valid = df[(df["TODO_date_column"] >= "TODO") & (df["TODO_date_column"] < "TODO")]
# test = df[df["TODO_date_column"] >= "TODO"]
# X_train, y_train = train[NUMERIC + CATEGORICAL], train[TARGET]
# X_valid, y_valid = valid[NUMERIC + CATEGORICAL], valid[TARGET]

In [10]:
# TODO today's rule and a naive baseline, scored on the validation data
# rule_pred = valid["TODO_column_the_rule_uses"]
# naive = DummyRegressor(strategy="median").fit(X_train, y_train)  # or DummyClassifier
# mean_absolute_error(y_valid, rule_pred), mean_absolute_error(y_valid, naive.predict(X_valid))

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 6. Candidate models

> 📝 At least **two** models besides the baselines: one simple (linear or logistic regression,
> a shallow tree) and one ensemble (random forest or gradient boosting). Build each as a
> pipeline, so the preprocessing is fitted on the training data only. Finish with one
> comparison table on the validation data, baselines included.

In [11]:
# TODO the preprocessing shared by every model
# preprocess = ColumnTransformer([
#     ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
#                       ("scale", StandardScaler())]), NUMERIC),
#     ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
# ])

In [12]:
# TODO the candidates
# candidates = {
#     "linear": Pipeline([("prep", preprocess), ("model", LinearRegression())]),
#     "random forest": Pipeline([("prep", preprocess),
#                                ("model", RandomForestRegressor(random_state=RANDOM_STATE))]),
# }

In [13]:
# TODO fit each one and build the comparison table
# rows = []
# for name, pipe in candidates.items():
#     pipe.fit(X_train, y_train)
#     pred = pipe.predict(X_valid)
#     rows.append({"model": name, "MAE": mean_absolute_error(y_valid, pred)})
# comparison = pd.DataFrame(rows)
# comparison

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 7. Metric and cost of errors

> 📝 Why is your main metric the right one for this decision? What does one error cost the
> stakeholder?
> - **Classification:** the confusion matrix read in business terms (what a false positive and
>   a false negative cost), and the threshold you chose.
> - **Regression:** MAE against RMSE, and where the large errors fall (which segments, which
>   periods).

In [14]:
# TODO the error analysis for the best candidate

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 8. Validation and tuning

> 📝 Cross-validate the chosen model (`TimeSeriesSplit` when your data has dates), tune it with
> a grid or random search, and only then evaluate it **once** on the test set. Report the
> score per fold, not only the mean: a model that is good on average but erratic is a risk.

In [15]:
# TODO cross-validation and the search
# cv = TimeSeriesSplit(n_splits=5)
# search = GridSearchCV(candidates["TODO"], param_grid={"model__max_depth": [4, 8, None]},
#                       cv=cv, scoring="neg_mean_absolute_error")
# search.fit(pd.concat([X_train, X_valid]), pd.concat([y_train, y_valid]))
# search.best_params_, -search.cv_results_["mean_test_score"]

In [16]:
# TODO the one final evaluation on the test set
# final_model = search.best_estimator_
# X_test, y_test = test[NUMERIC + CATEGORICAL], test[TARGET]
# mean_absolute_error(y_test, final_model.predict(X_test))

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 9. Interpretation

> 📝 Which features drive the predictions? Use permutation importance (or SHAP if you want to
> go further). Then explain two or three individual predictions: why did the model say this for
> this row? Do the drivers match your hypotheses?

In [17]:
# TODO feature importance
# result = permutation_importance(final_model, X_test, y_test, n_repeats=10,
#                                 random_state=RANDOM_STATE)
# importance = pd.Series(result.importances_mean, index=X_test.columns).sort_values()
# px.bar(importance, orientation="h", title="TODO")

In [18]:
# TODO two or three individual predictions, explained

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 10. Fairness, ethics and risks

> 📝 Compare the errors across at least one meaningful group (district, segment, period): is the
> model worse for some of them? Then answer in prose:
> - Does the data contain personal or sensitive information? How is it protected (lesson 03:
>   GDPR, anonymisation, access)?
> - What happens if the model is wrong, and who bears the cost?
> - One mitigation for each risk you name.

In [19]:
# TODO the error per group
# errors = test.assign(abs_error=(y_test - final_model.predict(X_test)).abs())
# errors.groupby("TODO_group")["abs_error"].mean()

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 11. Recommendation

> 📝 Speak to the stakeholder:
> - What should they do with the model, starting when?
> - The expected improvement over today's rule, translated into a KPI of your brief or into
>   money.
> - When it should **not** be trusted (periods, segments, data changes).
> - What block 4 adds: how it will be deployed and monitored.

*TODO*

## 12. Saving the model

> 📝 Save the final pipeline — from raw columns to a prediction — and prove it works by reloading
> it and predicting one row. This is the file lesson 10 deploys. Commit it if it is under 50 MB.

In [20]:
# TODO save, reload, predict one row
# joblib.dump(final_model, MODEL_PATH)
# reloaded = joblib.load(MODEL_PATH)
# reloaded.predict(X_test.head(1))

## 13. References

> 📝 Every source you used — the dataset, articles, documentation — in **APA** format. A missing
> or incorrect reference can cost the Pass.

- *TODO* Publisher. (Year). *Dataset title* [Data set]. URL